# COde 02 — ClinDistill: KD thành phần và đối chứng

Gắn bank v5/source 0.5.1 + cache v3, chọn GPU, tắt Internet, Run All.
Không đọc raw CSV, chia lại split hoặc mở test. DINO/BGE cố định, chỉ học MIL/fusion/head.
Mặc định 12 hệ thống × 3 seeds + 30 OOF fits = 66 fits.
CONFIG_FILE='research_controls.json' thêm single/mean MIL và clinical controls: 81 fits.
Các gate là giả thuyết cần kiểm nghiệm, không phải chứng nhận visual sufficiency.


In [ ]:
from pathlib import Path
import os, sys, json
RESEARCH_BANK_OVERRIDE = None
PROJECT_ROOT_OVERRIDE = None
FEATURE_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None
PREVIOUS_STUDY_OVERRIDE = None
CONFIG_FILE = 'research.json'
RUN_NAME = None  # dùng run_name của config; đổi khi thay protocol
RUN_TRAIN = True


## 1. Cài wheel offline và tìm cache

Dùng bank v5/source 0.5.1. Không trộn bank/cache/checkpoint cũ.


In [ ]:
from pathlib import Path
import os, sys, json, importlib.util
RESEARCH_BANK_OVERRIDE = globals().get('RESEARCH_BANK_OVERRIDE')
RESEARCH_ASSETS = None

def research_find(predicate, *, include_runs=False):
    found = set()
    for base in (Path.cwd(), Path.cwd().parent / 'Data', Path('/kaggle/input')):
        if not base.exists():
            continue
        for current, dirs, files in os.walk(base):
            here = Path(current)
            if predicate(here, dirs, files):
                found.add(here.resolve())
            excluded = {'Images', 'images', '.git', '.cache', '__pycache__', '_archive', '_support',
                        'models', 'wheels', 'wheelhouse', 'official_code', '_chunks', 'student_export'}
            dirs[:] = [d for d in dirs if not d.lower().startswith('qwen')]
            if not include_runs:
                excluded.add('runs')
            dirs[:] = [d for d in dirs if d not in excluded]
            if len(here.relative_to(base).parts) >= 8:
                dirs[:] = []
    return sorted(found, key=str)

def research_one(override, found, name):
    if override is not None:
        return Path(override).resolve()
    if len(found) != 1:
        raise RuntimeError(f'{name}: found {len(found)} folders: {found}. Set the matching *_OVERRIDE.')
    return found[0]

if Path('/kaggle').exists():
    bank_root = research_one(RESEARCH_BANK_OVERRIDE,
        research_find(lambda p, d, f: 'research_bank_manifest.json' in f), 'Research bank')
    helper = bank_root / 'source/_support/research_bank.py'
    spec = importlib.util.spec_from_file_location('research_bank_setup', helper)
    bank = importlib.util.module_from_spec(spec); spec.loader.exec_module(bank)
    RESEARCH_ASSETS = bank.activate(bank_root)
    PROJECT_ROOT = RESEARCH_ASSETS['source_root']
    # Compatibility for existing data-audit publishing cells only.
    OFFLINE_ASSETS = {'root': bank_root, 'source_root': PROJECT_ROOT}
else:
    PROJECT_ROOT = research_one(globals().get('PROJECT_ROOT_OVERRIDE'),
        research_find(lambda p, d, f: (p / 'clindistill/research').is_dir() and 'configs' in d), 'Research source')
    sys.path.insert(0, str(PROJECT_ROOT))
print('Research source:', PROJECT_ROOT)


In [ ]:
from clindistill.artifacts import read_json
FEATURE_ROOT = research_one(FEATURE_ROOT_OVERRIDE,
    research_find(lambda p,d,f: 'feature_manifest.json' in f and
        not read_json(p / 'feature_manifest.json').get('debug') and
        {'train.npz','validation.npz'} <= set(f)), 'Research feature cache')
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE) if OUTPUT_ROOT_OVERRIDE else (
    Path('/kaggle/working') if Path('/kaggle/working').exists() else PROJECT_ROOT / '_support')
CONFIG = read_json(PROJECT_ROOT / 'configs/code' / CONFIG_FILE)
RUN_NAME = RUN_NAME or CONFIG['run_name']
CONFIG['run_name'] = RUN_NAME
STUDY_ROOT = OUTPUT_ROOT / 'runs' / RUN_NAME
print('Features:', FEATURE_ROOT, '\nStudy:', STUDY_ROOT)


## 2. Preflight và resume

Kiểm tra checksum, nhóm bệnh nhân/ảnh trùng và folds trước khi train.
Để nối phiên trước: gắn Output cũ, đặt PREVIOUS_STUDY_OVERRIDE tới study folder.
Lưu optimizer/scheduler/RNG ở cuối mỗi epoch; epoch bị ngắt giữa chừng sẽ được chạy lại.

In [ ]:
import shutil, torch
from clindistill.research.data import FeatureStore
from clindistill.research.training import validate_config, grouped_folds, study_plan, run_study
validate_config(CONFIG)
if not torch.cuda.is_available() and not CONFIG['debug']:
    raise RuntimeError('Chọn Settings > Accelerator > GPU rồi chạy lại từ đầu.')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
else:
    print('CPU DEBUG: software verification only.')
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
STORE = FeatureStore(FEATURE_ROOT, allow_debug=CONFIG['debug'])
for seed in CONFIG['seeds']:
    grouped_folds(STORE.data['train']['labels'], STORE.data['train']['groups'], CONFIG['folds'], seed)
print(json.dumps(study_plan(CONFIG), indent=2))
print('Clinical policy:', STORE.metadata['request']['policy'])
if PREVIOUS_STUDY_OVERRIDE:
    if STUDY_ROOT.exists():
        raise RuntimeError('Study đã tồn tại: để PREVIOUS_STUDY_OVERRIDE=None để resume tại chỗ.')
    shutil.copytree(Path(PREVIOUS_STUDY_OVERRIDE), STUDY_ROOT)

## 3. Huấn luyện

Visual/MM Teachers → group-OOF → Students. Có label smoothing, correctness-only,
confidence-only, visual-support-only và reliability KD. OOF không fit nhóm held-out.
Weight chia mean toàn OOF train; CE vẫn học trên ca có gate=0.
_oof/seed_*/weight_diagnostics.json báo effective cases, zero fraction theo lớp/nhóm Teacher.
Clinical shuffle chỉ lấy context từ train của lần fit; validation/test giữ clinical tự nhiên.


In [ ]:
if RUN_TRAIN:
    validation_summary = run_study(FEATURE_ROOT, STUDY_ROOT, CONFIG, require_gpu=not CONFIG['debug'])
    display(validation_summary)
else:
    print('RUN_TRAIN=False: chưa train.')

## 4. Kết quả validation và learning curves

Chọn best checkpoint bằng macro-F1 validation. Test nằm riêng ở notebook03.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
if (STUDY_ROOT / 'validation_results.csv').exists():
    results = pd.read_csv(STUDY_ROOT / 'validation_results.csv'); display(results)
    methods = list(results.method.unique())
    rows = (len(methods) + 2) // 3
    fig, axes = plt.subplots(rows, 3, figsize=(16, 3.7 * rows), squeeze=False)
    for ax, method in zip(axes.flat, methods):
        for seed in CONFIG['seeds']:
            h = pd.read_csv(STUDY_ROOT / method / f'seed_{seed}' / 'history.csv')
            ax.plot(h.epoch, h.validation_macro_f1, label=str(seed))
        ax.set(title=method, xlabel='Epoch', ylabel='Validation macro-F1'); ax.legend()
    for ax in list(axes.flat)[len(methods):]: ax.set_visible(False)
    fig.tight_layout(); fig.savefig(STUDY_ROOT / 'learning_curves.png', dpi=160); plt.show()


## 5. Xuất Student chỉ ảnh

Method reliability đã đăng ký; chọn seed bằng validation. Gói export gồm head + DINO-B encoder local,
không cần clinical khi inference. Giữ weights/output riêng tư theo điều khoản model/dataset.

In [ ]:
from clindistill.research.reporting import export_student
if RUN_TRAIN and (STUDY_ROOT / 'completed.json').exists():
    encoder = RESEARCH_ASSETS['models']['student'] if RESEARCH_ASSETS else Path(STORE.metadata['model_locations']['student'])
    info = export_student(FEATURE_ROOT, STUDY_ROOT, encoder, OUTPUT_ROOT / 'student_export' / RUN_NAME)
    print(json.dumps(info, ensure_ascii=False, indent=2))
print('Save Version giữ runs/ và student_export/. Notebook03 đọc Output này để đánh giá.')